In [1]:
import transformers
import os
import pandas as pd

import torch

In [2]:
model_name = "meta-llama/Llama-2-13b-hf"
trec_path = "/home/guidorocchietti/llms/data/trec/"

### Pipeline

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
pipeline = transformers.pipeline(
        "text-generation",
        model=model_name,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto",
    )

### Partial model

In [3]:
df = pd.read_csv(os.path.join(trec_path, "trec19_for_rewriting.csv"))
prompts = df.prompt_rewritten_context.tolist()[:]

In [4]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
model = transformers.AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            # config=model_config,
            device_map="auto",
        )

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

#### Warming up the GPU
The experimental methodology was mainly inherited by https://deci.ai/blog/measure-inference-time-deep-neural-networks/, focusing in measuring latency. 

In [6]:
dummy_text = "Tyger Tyger, burning bright, In the forests of the night; What immortal hand or eye, Could frame thy fearful symmetry?"
tokenized_dummy_text = tokenizer(dummy_text, return_tensors="pt", padding=True).to("cuda:0")
for _ in range(10):
    _ = model.generate(**tokenized_dummy_text, max_length=48, temperature=0.7, repetition_penalty=1.1)

In [ ]:
total_time = 0
starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
for prompt in prompts:
    current_input = tokenizer(prompt, return_tensors="pt", truncation=True).to("cuda:0")
    with torch.no_grad():
        starter.record()
        output = model.generate(**current_input, max_length=48, temperature=0.7, repetition_penalty=1.1)
        ender.record()
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        total_time += curr_time
    

In [14]:
total_time / len(prompts)


345.2559841795464